# 01 · 環境檢查

| 小節 | 內容 |
|------|------|
| 1 | 路徑設定 |
| 2 | GPU / CUDA / ORT 驗證 |
| 3 | ONNX 模型檢視 |

## 1. 路徑設定
修改下方常數以符合本機環境。

In [1]:
from pathlib import Path
import shutil

_repo        = Path("../..").resolve()
_env_example = _repo / ".env.example"
_env_file    = _repo / ".env"

if not _env_file.exists():
    if _env_example.exists():
        shutil.copy(_env_example, _env_file)
        print(f"已建立 .env  ←  {_env_example.name}")
        print("請在 .env 中更新路徑後再繼續執行。")
    else:
        print(f"WARNING: 找不到 {_env_example}")
else:
    print(f".env 已存在：{_env_file}")

.env 已存在：D:\tensorrt\.env


In [ ]:
import sys, importlib
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve()))

import src.env
importlib.reload(src.env)
from src.env import *

ENGINES_DIR.mkdir(exist_ok=True)
print(f"TRT lib in PATH : {TRT_LIB}")
print(f"TEST_DATASET    : {TEST_DATASET}")
print("Settings ready.")

## 2. 環境驗證

In [3]:
import sys
import torch

print(f"Python          : {sys.version.split()[0]}")
print(f"PyTorch         : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU             : {props.name}")
    print(f"VRAM            : {props.total_memory / 1e9:.1f} GB")
    print(f"CUDA version    : {torch.version.cuda}")

print()
checks = [
    ("trtexec",       TRTEXEC),
    ("ONNX model",    ONNX_MODEL),
    ("Test dataset",  TEST_DATASET),
]
for label, path in checks:
    ok = path.exists()
    print(f"  {'[OK]    ' if ok else '[MISSING]'} {label}: {path}")

import onnxruntime as ort
print(f"\nONNXRuntime     : {ort.__version__}")
print(f"ORT providers   : {ort.get_available_providers()}")

Python          : 3.12.13
PyTorch         : 2.12.0+cu126
CUDA available  : True
GPU             : NVIDIA GeForce RTX 5070 Laptop GPU
VRAM            : 8.5 GB
CUDA version    : 12.6

  [OK]     trtexec: C:\Users\edisonhsieh\Downloads\TensorRT-10.8.0.43.Windows.win10.cuda-12.8\TensorRT-10.8.0.43\bin\trtexec.exe
  [OK]     ONNX model: C:\GPM_AI\H.onnx
  [OK]     Test dataset: C:\Users\edisonhsieh\Downloads\Test_dataset

ONNXRuntime     : 1.26.0
ORT providers   : ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']


d:\tensorrt\.venv\Lib\site-packages\torch\cuda\__init__.py:384: UserWarning: Found GPU0 NVIDIA GeForce RTX 5070 Laptop GPU which is of compute capability (CC) 12.0.
The following list shows the CCs this version of PyTorch was built for and the hardware CCs it supports:
- 5.0 which supports hardware CC >=5.0,<6.0 except {5.3}
- 6.0 which supports hardware CC >=6.0,<7.0 except {6.2}
- 6.1 which supports hardware CC >=6.1,<7.0 except {6.2}
- 7.0 which supports hardware CC >=7.0,<8.0 except {7.2}
- 7.5 which supports hardware CC >=7.5,<8.0
- 8.0 which supports hardware CC >=8.0,<9.0 except {8.7}
- 8.6 which supports hardware CC >=8.6,<9.0 except {8.7}
- 9.0 which supports hardware CC >=9.0,<10.0
Please follow the instructions at https://pytorch.org/get-started/locally/ to install a PyTorch release that supports one of these CUDA versions: 13.0, 13.2
  _warn_unsupported_code(d, device_cc, code_ccs)
d:\tensorrt\.venv\Lib\site-packages\torch\cuda\__init__.py:502: UserWarning: 
NVIDIA GeForce 

In [ ]:
import onnxruntime as ort

# Verify CUDA session actually initialises (not just listed as available)
try:
    _sess = ort.InferenceSession(
        str(ONNX_MODEL),
        providers=["CUDAExecutionProvider"]
    )
    print(f"ORT CUDA session : OK  (active: {_sess.get_providers()[0]})")
except Exception as e:
    print(f"ORT CUDA session : FAILED — {e}")

# Count test images so downstream notebooks don't silently get 0 results
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
_imgs = [p for p in TEST_DATASET.rglob("*") if p.suffix.lower() in IMG_EXTS and p.is_file()]
print(f"Test images found: {len(_imgs)}  in {TEST_DATASET}")
if not _imgs:
    print("  WARNING: no images found — inference cells in notebook 02 will be skipped.")

## 3. ONNX 模型檢視
確認輸入/輸出 shape 與 opset，避免後續轉換出錯。

In [4]:
import onnx

model = onnx.load(str(ONNX_MODEL))
onnx.checker.check_model(model)

print(f"IR version : {model.ir_version}")
print(f"Opset      : {model.opset_import[0].version}")

def fmt_shape(tensor_type):
    dims = tensor_type.shape.dim
    return [d.dim_value if d.dim_value > 0 else (d.dim_param or "?") for d in dims]

print("\nInputs:")
for t in model.graph.input:
    print(f"  {t.name:40s}  {fmt_shape(t.type.tensor_type)}")

print("\nOutputs:")
for t in model.graph.output:
    print(f"  {t.name:40s}  {fmt_shape(t.type.tensor_type)}")

print(f"\nTotal graph nodes: {len(model.graph.node)}")

_in    = model.graph.input[0]
_shape = fmt_shape(_in.type.tensor_type)
MODEL_H = _shape[2] if isinstance(_shape[2], int) else 448
MODEL_W = _shape[3] if isinstance(_shape[3], int) else 448
MODEL_C = _shape[1] if isinstance(_shape[1], int) else 3
print(f"\nUsing inference shape: [1, {MODEL_C}, {MODEL_H}, {MODEL_W}]")

IR version : 7
Opset      : 12

Inputs:
  images                                    [1, 3, 448, 448]

Outputs:
  output                                    [1, 6]

Total graph nodes: 122

Using inference shape: [1, 3, 448, 448]
